In [1]:
%reload_ext autoreload
%autoreload 2

import os
from pathlib import Path

print(Path().cwd())
os.chdir(Path(os.getcwd()).parent)
print(Path().cwd())

/Users/samantha/QuantUS-Plugins-CEUS/TwoD_CEUS_test
/Users/samantha/QuantUS-Plugins-CEUS


## Select Contrast-Enhanced Ultrasound (CEUS) Cine and Parser

In [2]:
from src.image_loading.options import get_scan_loaders

print("Available scan loaders:", list(get_scan_loaders().keys()))

Available scan loaders: ['avi', 'nifti', 'custom_dicom', 'mp4']


In [31]:
scan_type = 'nifti'

scan_path = '/Volumes/Extreme Pro/UCSD_SSD/ChinaData/3D-002_3rd/CEUS-25461/CEUS-25461.nii.gz'
scan_loader_kwargs = {
    'transpose': False,
}

In [32]:
from src.entrypoints import scan_loading_step

image_data = scan_loading_step(scan_type, scan_path, **scan_loader_kwargs)

## Load Segmentation

Assumes same segmentation for each frame

In [33]:
from src.seg_loading.options import get_seg_loaders

print("Available segmentation loaders:", list(get_seg_loaders().keys()))

Available segmentation loaders: ['nifti', 'load_bolus_mask']


In [34]:
seg_type = 'nifti'

seg_path = '/Users/samantha/Desktop/ultrasound lab stuff/china data/p2/v3/voi_necrotic_removed.nii.gz'
seg_loader_kwargs = {}

In [35]:
from src.entrypoints import seg_loading_step

seg_data = seg_loading_step(seg_type, image_data, seg_path, scan_path, **seg_loader_kwargs)

## CEUS Quantitative Temporal Curve Analysis (Parametric Map Mode)

In [36]:
from src.time_series_analysis.options import get_analysis_types, get_required_kwargs

all_analysis_types, all_analysis_funcs = get_analysis_types()
print("Available analysis types:", list(all_analysis_types.keys()))

Available analysis types: ['curves_paramap', 'curves']


In [37]:
# IMPORTANT: Use curves_paramap for parametric map generation
analysis_type = 'curves_paramap'

print("Available analysis functions:", list(all_analysis_funcs.keys()))

Available analysis functions: ['pyradiomics', 'tic']


In [38]:
analysis_funcs = ['tic']

required_kwargs = get_required_kwargs(analysis_type, analysis_funcs)
print("Required kwargs for current analysis:", required_kwargs)

Required kwargs for current analysis: ['sag_vox_ovrlp', 'ax_vox_ovrlp', 'cor_vox_ovrlp', 'sag_vox_len', 'cor_vox_len', 'ax_vox_len']


In [39]:
# Set frame rate
image_data.frame_rate = 1

# Required kwargs for parametric map analysis
analysis_kwargs = {
    'ax_vox_ovrlp': 50,
    'sag_vox_ovrlp': 50,
    'cor_vox_ovrlp': 50,
    'ax_vox_len': 20.0,
    'sag_vox_len': 20.0,
    'cor_vox_len': 20.0,
}

In [40]:
from src.entrypoints import analysis_step

analysis_obj = analysis_step(analysis_type, image_data, seg_data, analysis_funcs, **analysis_kwargs)

# Verify we got the right type
print("Analysis object type:", type(analysis_obj))

Computing curves: 100%|██████████| 216/216 [02:36<00:00,  1.38it/s]

Analysis object type: <class 'src.time_series_analysis.curves_paramap.framework.CurvesParamapAnalysis'>


## Curve Quantification

In [41]:
from src.curve_quantification.options import get_quantification_funcs

quantification_funcs = get_quantification_funcs()
print("Available quantification functions:", quantification_funcs.keys())

Available quantification functions: dict_keys(['auc_no_fit', 'cmus_firstorder', 'dte', 'first_order_full', 'first_order_select', 'lognormal_fit_full', 'lognormal_fit_select', 'wash_rates'])


In [42]:
function_names = ['lognormal_fit_full']  # or [] for all functions
output_path = '/Users/samantha/Desktop/ultrasound lab stuff/china data/p2/v3/paramap3/output.csv'
curve_quantifications_kwargs = {
    'curves_to_fit': ['TIC'],
    'tic_name': 'TIC'
}

In [43]:
from src.entrypoints import curve_quantification_step

curve_quant = curve_quantification_step(analysis_obj, function_names, output_path, **curve_quantifications_kwargs)

# Verify analysis type
print("curve_quant.analysis_objs type:", type(curve_quant.analysis_objs))

curve_quant.analysis_objs type: <class 'src.time_series_analysis.curves_paramap.framework.CurvesParamapAnalysis'>


## Parametric Map Saving

In [44]:
from src.entrypoints import visualization_step

vis_type = 'paramap'
params = []
vis_funcs = []
vis_kwargs = {
    'paramap_folder_path': '/Users/samantha/Desktop/ultrasound lab stuff/china data/p2/v3/paramap3',
    'hide_all_visualizations': False,
}

vis_obj = visualization_step(curve_quant, vis_type, params, vis_funcs, **vis_kwargs)